Dataset

In [18]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import json
import os
import time
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional

# ==========================================
# 1. DATA STRUCTURES & SCHEMAS
# ==========================================

@dataclass
class MedicalCase:
    case_id: str
    dataset_name: str
    case_text: str
    options: Optional[Dict[str, str]]  # e.g., {"A": "...", "B": "..."}
    evidence_context: Optional[str]    # e.g., PubMed Abstract
    gold_label: str

@dataclass
class AgentOutput:
    final_answer: str
    confidence: float
    diagnosis_or_hypothesis: List[str]
    reasoning: str
    cited_evidence: List[str]
    missing_evidence: List[str]
    safety_concerns: List[str]
    revision_note: Optional[str] = None

@dataclass
class ExecutionTrace:
    case_id: str
    architecture: str
    round_1_outputs: Dict[str, Any]  # Maps agent_id -> parsed schema
    round_2_outputs: Dict[str, Any]  # Maps agent_id -> parsed schema (Empty for CoT)
    aggregated_answer: str
    total_tokens_used: int
    estimated_cost: float

# ==========================================
# 2. LLM CLIENT WRAPPER
# ==========================================

class MedicalLLMClient:
    def __init__(self, model_name: str = "gpt-4o", temperature: float = 0.4):
        self.model_name = model_name
        self.temperature = temperature
        # Real pricing: gpt-4o-2024-05-13 approx values
        self.input_token_cost = 5.00 / 1000000
        self.output_token_cost = 15.00 / 1000000

    def _get_system_prompt(self) -> str:
        return """You are an expert, board-certified clinical AI assistant.
Your analysis must be evidence-grounded, rigorous, and highly attentive to patient safety.
You MUST respond strictly with a valid JSON object matching the requested schema. Do not include markdown formatting like ```json or any trailing text."""

    def _get_schema_instructions(self) -> str:
        return """
Output MUST strictly follow this JSON structure:
{
  "final_answer": "Your final chosen option",
  "confidence": 0.85,
  "diagnosis_or_hypothesis": ["Primary Diagnosis"],
  "reasoning": "Clinical rationale...",
  "cited_evidence": ["Fact from case"],
  "missing_evidence": ["Required data"],
  "safety_concerns": ["Risks"],
  "revision_note": null
}"""

    def call_agent(self, user_prompt: str, agent_role: str = "Generalist Clinical Agent") -> tuple[Dict[str, Any], int]:
        full_system_prompt = f"{self._get_system_prompt()}\nYour Role: {agent_role}\n{self._get_schema_instructions()}"

        # Use the real API call logic
        try:
            parsed_output, tokens = call_real_llm(full_system_prompt, user_prompt, model=self.model_name)
            return parsed_output, tokens
        except Exception as e:
            print(f"API Error, falling back to mock: {e}")
            # Fallback mock for stability if API fails
            mock_resp = {"final_answer": "A", "confidence": 0.0, "reasoning": "Error Fallback"}
            return mock_resp, 0

# ==========================================
# 3. ARCHITECTURE PIPELINES
# ==========================================

class AuditPipelineOrchestrator:
    def __init__(self, client: MedicalLLMClient):
        self.client = client

    def _build_case_prompt(self, case: MedicalCase) -> str:
        prompt = f"### CLINICAL CASE CONTEXT\n{case.case_text}\n\n"
        if case.evidence_context:
            prompt += f"### GROUNDING EVIDENCE/ABSTRACT\n{case.evidence_context}\n\n"
        if case.options:
            prompt += f"### OPTIONS\n"
            for key, val in case.options.items():
                prompt += f"{key}: {val}\n"
        return prompt

    def run_single_agent_cot(self, case: MedicalCase) -> ExecutionTrace:
        user_prompt = self._build_case_prompt(case)
        user_prompt += "\nPerform a systematic clinical evaluation. Detail your reasoning and commit to a final answer."
        parsed_json, tokens = self.client.call_agent(user_prompt, agent_role="Single-Agent CoT")
        cost = tokens * self.client.output_token_cost
        return ExecutionTrace(case.case_id, "Single-Agent CoT", {"Agent_Primary": parsed_json}, {}, parsed_json.get("final_answer", "UNKNOWN"), tokens, cost)

    def independent_vote(self, case: MedicalCase) -> ExecutionTrace:
        agent_ids = ["Agent_Alpha", "Agent_Beta", "Agent_Gamma"]
        results = {}
        total_tokens = 0 # Initialize total_tokens
        base_case_prompt = self._build_case_prompt(case)
        r1_prompt= base_case_prompt + "\nPerform a systematic clinical evaluation. Detail your reasoning and commit to a final answer."

        for agent in agent_ids:
            out, tokens = self.client.call_agent(r1_prompt, agent_role=f"Symmetric Debater ({agent})")
            results[agent] = out
            total_tokens += tokens
        votes = [results[a].get("final_answer") for a in agent_ids]
        final_vote= max(set(votes), key=votes.count)
        cost= total_tokens*self.client.output_token_cost
        return ExecutionTrace(case.case_id, "Independent Vote", results, {}, final_vote, total_tokens, cost)

    def run_symmetric_debate(self, case: MedicalCase) -> ExecutionTrace:
        agent_ids = ["Agent_Alpha", "Agent_Beta", "Agent_Gamma"]
        base_case_prompt = self._build_case_prompt(case)
        total_tokens = 0
        r1_results, r2_results = {}, {}

        r1_prompt = base_case_prompt + "\nAnalyze the case independently."
        for agent in agent_ids:
            out, tokens = self.client.call_agent(r1_prompt, agent_role=f"Symmetric Debater ({agent})")
            r1_results[agent] = out
            total_tokens += tokens

        peer_insights = "### ANONYMOUS PEER RATIONALES\n"
        for p_id, p_data in r1_results.items():
            peer_insights += f"- Answer: {p_data.get('final_answer')}. Rationale: {p_data.get('reasoning')}\n"

        for agent in agent_ids:
            r2_prompt = base_case_prompt + f"\n{peer_insights}\nCritique peer perspectives and finalize your answer."
            out, tokens = self.client.call_agent(r2_prompt, agent_role=f"Symmetric Debater ({agent})")
            r2_results[agent] = out
            total_tokens += tokens

        r2_votes = [r2_results[a].get("final_answer") for a in agent_ids]
        final_agg = max(set(r2_votes), key=r2_votes.count)
        cost = total_tokens * self.client.output_token_cost
        return ExecutionTrace(case.case_id, "Symmetric Debate", r1_results, r2_results, final_agg, total_tokens, cost)

def save_trace_to_jsonl(trace: ExecutionTrace, output_path: str):
    with open(output_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(asdict(trace)) + "\n")

# New Section

In [20]:
import openai
from google.colab import userdata
import json

# OpenRouter configuration
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
OR_CLIENT = openai.OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=OPENROUTER_API_KEY,
)

def call_real_llm(system_prompt, user_prompt, model="openai/gpt-4o-mini"):
    """
    Implementation for OpenRouter API calls.
    Defaulting to a cost-effective model, but you can change this to any
    OpenRouter supported ID (e.g., 'openai/gpt-4o').
    """
    response = OR_CLIENT.chat.completions.create(
      extra_headers={
        "HTTP-Referer": "https://colab.research.google.com/", # Optional, for OpenRouter rankings
        "X-Title": "Medical Agent Audit", # Optional
      },
      model=model,
      messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
      ],
      response_format={ "type": "json_object" },
      temperature=0.4
    )

    raw_text = response.choices[0].message.content
    usage = response.usage.total_tokens
    return json.loads(raw_text), usage

print("OpenRouter client initialized. Your MedicalLLMClient will now use this function.")

OpenRouter client initialized. Your MedicalLLMClient will now use this function.


In [21]:
from google.colab import drive
import pandas as pd
import numpy as np
import json

# Mount drive

def load_qa_dataset(path='/content/drive/MyDrive/QA_data.json'):
    """Loads data from your specific JSON format."""
    with open(path, 'r') as f:
        data = json.load(f)

    cases = []
    for item in data:
        # Your format uses 'answer' string like 'A. IGF1', we need the index 'A'
        gold_label = item.get('answer', ' ')[0]

        cases.append(MedicalCase(
            case_id=str(item.get('Index')),
            dataset_name="Drive_QA",
            case_text=item.get('question'),
            options=None, # Options are already in the question text for this format
            evidence_context=" ".join(item.get('Scoring_Points', [])),
            gold_label=gold_label
        ))
    return cases

In [22]:
class MedicalAuditEvaluator:
    """
    Implementation of Core Metrics for Multi-Agent Clinical Analysis.
    """
    def __init__(self):
        self.results = []

    def calculate_metrics(self, trace: ExecutionTrace, gold_label: str):
        """Processes a single trace to compute metrics."""
        # 1. Final Accuracy
        is_correct = trace.aggregated_answer.strip().startswith(gold_label)

        # 2. Answer Agreement (Round 1 vs Round 2)
        r1_answers = [out.get('final_answer', '') for out in trace.round_1_outputs.values()]
        r2_answers = [out.get('final_answer', '') for out in trace.round_2_outputs.values()]

        def get_agreement(answers):
            if not answers: return 1.0
            pairs = 0
            matches = 0
            for i in range(len(answers)):
                for j in range(i + 1, len(answers)):
                    pairs += 1
                    if answers[i] == answers[j]: matches += 1
            return matches / pairs if pairs > 0 else 1.0

        agreement_r1 = get_agreement(r1_answers)
        agreement_r2 = get_agreement(r2_answers) if r2_answers else agreement_r1

        # 3. Evidence Overlap (Simple F1 between agents)
        all_evidence = [set(out.get('cited_evidence', [])) for out in trace.round_1_outputs.values()]
        # Simplified: Average pairwise Jaccard similarity as overlap proxy
        overlaps = []
        for i in range(len(all_evidence)):
            for j in range(i + 1, len(all_evidence)):
                union = len(all_evidence[i] | all_evidence[j])
                intersection = len(all_evidence[i] & all_evidence[j])
                overlaps.append(intersection / union if union > 0 else 1.0)
        avg_evidence_overlap = np.mean(overlaps) if overlaps else 1.0

        # 4. Revision Utility
        # Utility = (Wrong -> Right) - (Right -> Wrong)
        utility = 0
        if trace.round_2_outputs:
            for agent_id in trace.round_1_outputs:
                r1_correct = trace.round_1_outputs[agent_id].get('final_answer', '').startswith(gold_label)
                r2_correct = trace.round_2_outputs[agent_id].get('final_answer', '').startswith(gold_label)
                if not r1_correct and r2_correct: utility += 1
                if r1_correct and not r2_correct: utility -= 1

        # 5. Confidence Calibration
        confidences = [out.get('confidence', 0) for out in trace.round_1_outputs.values()]
        avg_conf = np.mean(confidences)

        metrics = {
            "case_id": trace.case_id,
            "correct": is_correct,
            "agreement_r1": agreement_r1,
            "agreement_r2": agreement_r2,
            "evidence_overlap": avg_evidence_overlap,
            "revision_utility": utility,
            "avg_confidence": avg_conf,
            "cost": trace.estimated_cost
        }
        self.results.append(metrics)
        return metrics

    def get_summary(self):
        df = pd.DataFrame(self.results)
        summary = {
            "Mean Accuracy": df['correct'].mean(),
            "Consensus Shift": df['agreement_r2'].mean() - df['agreement_r1'].mean(),
            "Avg Evidence Overlap": df['evidence_overlap'].mean(),
            "Total Revision Utility": df['revision_utility'].sum(),
            "Total Cost": df['cost'].sum()
        }
        return summary

In [23]:
# Consolidated Execution Loop with Evaluators
try:
    cases = load_qa_dataset()[:2] # Testing first 3 cases
    llm_client = MedicalLLMClient(model_name="openai/gpt-4o-mini")
    orchestrator = AuditPipelineOrchestrator(llm_client)

    # Initialize separate evaluators for each architecture
    debate_evaluator = MedicalAuditEvaluator()
    independent_evaluator = MedicalAuditEvaluator()
    cot_evaluator = MedicalAuditEvaluator()

    print("--- Running Symmetric Debate ---")
    for case in cases:
        print(f"Processing Case {case.case_id} with Symmetric Debate...")
        trace = orchestrator.run_symmetric_debate(case)
        debate_evaluator.calculate_metrics(trace, case.gold_label)

    print("\n--- Running Independent Vote ---")
    for case in cases:
        print(f"Processing Case {case.case_id} with Independent Vote...")
        trace = orchestrator.independent_vote(case)
        independent_evaluator.calculate_metrics(trace, case.gold_label)

    print("\n--- Running Single-Agent CoT ---")
    for case in cases:
        print(f"Processing Case {case.case_id} with Single-Agent CoT...")
        trace = orchestrator.run_single_agent_cot(case)
        cot_evaluator.calculate_metrics(trace, case.gold_label)

    # Display results for Symmetric Debate
    print("\n### Symmetric Debate Results")
    debate_results_df = pd.DataFrame(debate_evaluator.results)
    display(debate_results_df)
    print("\n--- AGGREGATED METRICS (Symmetric Debate) ---")
    print(debate_evaluator.get_summary())

    # Display results for Independent Vote
    print("\n### Independent Vote Results")
    independent_results_df = pd.DataFrame(independent_evaluator.results)
    display(independent_results_df)
    print("\n--- AGGREGATED METRICS (Independent Vote) ---")
    print(independent_evaluator.get_summary())

    # Display results for Single-Agent CoT
    print("\n### Single-Agent CoT Results")
    cot_results_df = pd.DataFrame(cot_evaluator.results)
    display(cot_results_df)
    print("\n--- AGGREGATED METRICS (Single-Agent CoT) ---")
    print(cot_evaluator.get_summary())

except Exception as e:
    print(f"Error: {e}. Please ensure the QA_data.json file exists and its path is correct in load_qa_dataset.")

--- Running Symmetric Debate ---
Processing Case 1 with Symmetric Debate...
Processing Case 2 with Symmetric Debate...
Processing Case 3 with Symmetric Debate...

--- Running Independent Vote ---
Processing Case 1 with Independent Vote...
Processing Case 2 with Independent Vote...
Processing Case 3 with Independent Vote...

--- Running Single-Agent CoT ---
Processing Case 1 with Single-Agent CoT...
Processing Case 2 with Single-Agent CoT...
Processing Case 3 with Single-Agent CoT...

### Symmetric Debate Results


,case_id,correct,agreement_r1,agreement_r2,evidence_overlap,revision_utility,avg_confidence,cost
0,1,True,1.0,1.0,0.000000,0,0.883333,0.057810
1,2,True,1.0,1.0,0.000000,0,0.850000,0.055905
2,3,True,1.0,1.0,0.333333,0,0.866667,0.051690



--- AGGREGATED METRICS (Symmetric Debate) ---
{'Mean Accuracy': np.float64(1.0), 'Consensus Shift': np.float64(0.0), 'Avg Evidence Overlap': np.float64(0.1111111111111111), 'Total Revision Utility': np.int64(0), 'Total Cost': np.float64(0.16540500000000002)}

### Independent Vote Results


,case_id,correct,agreement_r1,agreement_r2,evidence_overlap,revision_utility,avg_confidence,cost
0,1,True,1.0,1.0,0.000000,0,0.866667,0.024045
1,2,True,1.0,1.0,0.000000,0,0.850000,0.024360
2,3,True,1.0,1.0,0.333333,0,0.883333,0.022275



--- AGGREGATED METRICS (Independent Vote) ---
{'Mean Accuracy': np.float64(1.0), 'Consensus Shift': np.float64(0.0), 'Avg Evidence Overlap': np.float64(0.1111111111111111), 'Total Revision Utility': np.int64(0), 'Total Cost': np.float64(0.07068)}

### Single-Agent CoT Results


,case_id,correct,agreement_r1,agreement_r2,evidence_overlap,revision_utility,avg_confidence,cost
0,1,True,1.0,1.0,1.0,0,0.95,0.008160
1,2,True,1.0,1.0,1.0,0,0.90,0.008085
2,3,True,1.0,1.0,1.0,0,0.90,0.007575



--- AGGREGATED METRICS (Single-Agent CoT) ---
{'Mean Accuracy': np.float64(1.0), 'Consensus Shift': np.float64(0.0), 'Avg Evidence Overlap': np.float64(1.0), 'Total Revision Utility': np.int64(0), 'Total Cost': np.float64(0.02382)}
